<a href="https://colab.research.google.com/github/sudenazgrmz/fake-review-robustness/blob/main/04_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score

df = pd.read_csv("deceptive-opinion.csv")
X = df["text"]
y = (df["deceptive"] == "deceptive").astype(int)
groups = df["hotel"]

def remove_words(texts, words):
    pattern = r"\b(" + "|".join(re.escape(w) for w in words) + r")\b"
    return texts.str.replace(pattern, " ", regex=True, flags=re.IGNORECASE)

source_words = ["priceline", "expedia", "orbitz", "tripadvisor", "yelp"]
place_words = ["chicago", "michigan", "hilton", "millennium"] + sorted(df["hotel"].unique())

variants = {
    "baseline": X,
    "no_source_names": remove_words(X, source_words),
    "no_place_and_hotel_names": remove_words(X, place_words),
    "no_both": remove_words(X, source_words + place_words),
}

# sanity check: how many reviews were changed by each removal
for name, texts in variants.items():
    print(name, "- changed reviews:", int((texts != X).sum()))
print()

models = {
    "tfidf_logreg": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2),
        LogisticRegression(max_iter=1000),
    ),
    "tfidf_svm": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2),
        LinearSVC(),
    ),
}

group_cv = GroupKFold(n_splits=5)

for model_name, model in models.items():
    print(model_name)
    for var_name, texts in variants.items():
        scores = cross_val_score(model, texts, y, cv=group_cv, groups=groups, scoring="f1_macro")
        print("  ", var_name.ljust(26), "macro-F1 mean", round(scores.mean(), 3), "std", round(scores.std(), 3))
    print()

baseline - changed reviews: 0
no_source_names - changed reviews: 55
no_place_and_hotel_names - changed reviews: 1268
no_both - changed reviews: 1284

tfidf_logreg
   baseline                   macro-F1 mean 0.87 std 0.015
   no_source_names            macro-F1 mean 0.869 std 0.014
   no_place_and_hotel_names   macro-F1 mean 0.862 std 0.007
   no_both                    macro-F1 mean 0.86 std 0.007

tfidf_svm
   baseline                   macro-F1 mean 0.884 std 0.018
   no_source_names            macro-F1 mean 0.882 std 0.015
   no_place_and_hotel_names   macro-F1 mean 0.871 std 0.014
   no_both                    macro-F1 mean 0.87 std 0.012

